# Tools, Resources and Prompts

**Prerequisites:** `01_mcp_concepts.ipynb`, `02_mcp_client.ipynb`. From `03_tools`: `03_invocation.ipynb`. Helpful: `04_memory`'s README

Notebook 1 asked the demo server for its resources and its prompts and got two empty lists back, from a server that had declared both capabilities. Notebook 2 then spent its whole length on tools, which is what almost every MCP server in the wild actually offers.

This notebook is about the other two, and it has to do something the previous two did not: **argue that they are worth having at all.** A tool can already return any bytes you like. `weather://stations` could have been `list_stations()`. So the question is not how to read a resource — that is one method call — but what the distinction buys, and when it buys nothing.

The short version, which the rest of the notebook tries to earn:

| | who decides it happens | when it costs tokens |
|---|---|---|
| **tool** | the model | when the model calls it |
| **resource** | the host | every turn it is included |
| **prompt** | a person | when they pick it |

Three different actors. That is the whole distinction, and everything awkward below follows from the third column.

## Setup

`server.py` is unchanged and still tools-only — notebook 1's finding depends on that, and a server which declares both capabilities and implements neither is worth keeping around, because it is what most of the ecosystem looks like.

So the data lives in a **second server**, `data_server.py`. That is also the more honest arrangement: a weather service and a document store are two services, usually written by two different people.

In [1]:
import json
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
sys.path.append(str(Path.cwd().parent.parent / "01_foundations"))
sys.path.append(str(Path.cwd().parent.parent / "03_tools"))

from chat import count_tokens
from invoke import invoke
from registry import Registry

from client import DATA_SERVER, SERVER
from remote import remote_tools, resource_reader
from session import Session

tools_only = Session(SERVER).__enter__()
session = Session(DATA_SERVER).__enter__()

print(f"{tools_only.server_name}: "
      f"{len(tools_only.list_tools())} tools, "
      f"{len(tools_only.list_resources())} resources, "
      f"{len(tools_only.list_prompts())} prompts")
print(f"{session.server_name}:  "
      f"{len(session.list_tools())} tools, "
      f"{len(session.list_resources())} resources, "
      f"{len(session.list_prompts())} prompts")

forge-demo: 6 tools, 0 resources, 0 prompts
forge-data:  0 tools, 2 resources, 1 prompts


## What a resource is

Not a name and a schema. A **URI** and a mime type.

In [2]:
for r in session.list_resources():
    print(f"{str(r.uri):22} {r.mime_type:18} {r.description}")

weather://stations     application/json   The stations this service has readings for.
weather://history      text/csv           Every reading ever taken. Deliberately far larger than a context window wants.


There is no `inputSchema`, because there are no arguments — you do not *call* `weather://stations`, you *read* it. That is the difference the whole primitive rests on, and it is a difference about initiative rather than about data.

## The trap: templates are not in that list

The server also offers a parameterised resource, and it did not appear above.

In [3]:
print("resources:", [str(r.uri) for r in session.list_resources()])
print("templates:", [t.uri_template for t in session.list_resource_templates()])

resources: ['weather://stations', 'weather://history']
templates: ['weather://station/{city}']


`resources/list` and `resources/templates/list` are **two different calls**, and the first does not include the second. A server offering nothing but templates answers `resources/list` with `[]`.

Which means notebook 1's finding was slightly worse than it looked. When a host checks `resources/list`, gets `[]`, and concludes the server has no data to offer, it may be wrong — and it will be wrong silently. A capability was not an inventory; now one inventory is not the inventory either.

## Reading one

In [4]:
got = session.read_resource("weather://stations")
block = got.contents[0]

print("uri:      ", block.uri)
print("mime_type:", block.mime_type)
print("text:     ", block.text)
print()
print("a template is read by filling the URI in:")
print(session.read_resource("weather://station/oslo").contents[0].text)

uri:       weather://stations
mime_type: application/json
text:      ["cairo", "oslo", "tokyo"]

a template is read by filling the URI in:
{"city": "oslo", "celsius": 4, "source": "forge-data"}


Notice what a template actually is: a URI with a hole in it, filled by the caller. Which is to say it takes an argument.

**So how is `weather://station/{city}` different from `weather(city)`?** Not in what it returns. Not in what it costs. The honest answer is that it is different only in who is expected to initiate it — and since the protocol cannot enforce that expectation, a parameterised resource is the place where the tool/resource distinction is at its blurriest. It is worth being suspicious of one. A resource earns the name when it is something a host would sensibly paste in unprompted; if the host would have to decide *which* one to read based on what the user just said, the model is the thing making that decision, and it wanted a tool.

## The column that matters: what it costs

Here is the resource the distinction actually bites on.

In [5]:
history = session.read_resource("weather://history").contents[0].text

print(f"weather://history is {len(history):,} characters")
print(f"                  = {count_tokens(history):,} tokens")
print()
print("first two rows:", history.splitlines()[:2])

weather://history is 21,239 characters
                  = 12,780 tokens

first two rows: ['date,city,celsius', '2026-02-02,tokyo,19']


**12,780 tokens.** A host that decides `weather://history` is relevant and pastes it into the context window pays that on the turn it does so — and on every subsequent turn, because it is now in the message list.

Compare what `03_tools` does with a large *tool* result. Notebook 3 of that chapter built `invoke(max_tokens=)` for exactly this, after a 7,601-token response: the whole value stays on `Result.value` and only what the conversation can afford goes into it.

Nothing does that for a resource. There is no size negotiation in `resources/read`, no `max_tokens`, no warning. The host asked, so the host gets all of it.

That is not an oversight in the protocol — it follows from the definition. A resource is host-pulled, and the host is assumed to know what it is doing. It does mean the safeguard chapter 3 built does not apply on this path, and that **a resource is a context-window decision wearing the costume of a data-access decision.** Which is `05_context`'s subject, arriving two chapters early.

## Giving the decision back to the model

There is an obvious move here, and most real hosts make it: expose reading as a tool.

In [6]:
reader = resource_reader(session)

print("name:       ", reader.name)
print("description:", reader.description)
print("schema:     ", json.dumps(reader.schema["function"]["parameters"]))

name:        read_resource
description: Read one of this server's resources by URI. Available: weather://stations, weather://history
schema:      {"additionalProperties": false, "properties": {"uri": {"type": "string"}}, "required": ["uri"], "type": "object"}


Now the model chooses, and something useful comes back with the choice:

In [7]:
r = invoke(reader, {"uri": "weather://history"}, max_tokens=150)

print("kind:     ", r.kind)
print("truncated:", r.truncated)
print(f"into the conversation: {count_tokens(r.content):,} tokens")
print(f"kept on Result.value:  {count_tokens(str(r.value)):,} tokens")

kind:      ok
truncated: True
into the conversation: 168 tokens
kept on Result.value:  12,780 tokens


168 tokens instead of 12,780, and the full text still reachable by the program. Going through a tool put the resource back under `invoke()`'s size limit, because `invoke()` does not care where bytes came from.

**Now look at what the model actually received**, because the number is more flattering than the result:

In [8]:
print(r.content[-200:])

3,oslo,6
2026-03-03,cairo,35
2026-04-04,tokyo,21
2026-04-04,oslo,7
2026-04-04,cairo,36
2026-05-05,tokyo,22
2026-05-05,oslo,8
2026-05-05,cairo,37
2026-06
[... cut short: 12,780 tokens in all, 150 kept]


The first thirty-odd rows of a CSV, and then a note saying it stopped. The budget was protected and **the data was destroyed** — there is no question about 2026 weather that those rows answer, because the rows it needed were somewhere in the other 12,600 tokens.

This is worth being blunt about, because "wrap it in a tool" reads like a solution and is only half of one. Truncation is a *safety limit*, not an access strategy. It stops one resource eating the window; it does not make a large resource usable. For anything corpus-sized the answer is not a smaller slice of the front — it is not fetching the whole thing in the first place, which is the retrieval section at the end of this notebook.

Where the tool wrapper genuinely does solve the problem is the small resource: `weather://stations` is a handful of tokens, the model can ask for it when it needs it, and nothing is lost.

So the trade, stated plainly:

- **host-pulled** — arrives reliably, costs its tokens every turn whether it was needed or not, and no size limit applies
- **model-pulled** — costs nothing until asked for, is trimmed on the way in, and depends on the model thinking to ask, which it frequently does not

Neither is correct in general. What settles it is whether the data is needed on *every* turn — a project's coding conventions, say — or on the rare turn where it happens to be relevant. `04_memory` reached the same fork and named it there: **recall by injection versus recall as a tool**, landing on injection for what must be dependable and a tool for what must be cheap. This is that decision again, one layer out, and the answer does not change.

One more thing the wrapper is doing, which is easy to miss:

In [9]:
r = invoke(reader, {"uri": "weather://nope"})
print(r.kind, "|", r.content)

refused | Unknown resource: weather://nope. Available: weather://stations, weather://history


A URI the server does not have arrives as a **protocol error**, not as `is_error` — a different channel again, and one notebook 2's mapping never saw, because tools do not use it. Left alone it lands in `invoke()`'s `except Exception` arm and is classified `broken`, which means `for_the_model` is False and the model never learns it guessed a bad URI.

That is precisely backwards: naming a resource that does not exist is the most fixable mistake there is, given the list. So `resource_reader` catches it and raises `ToolError`, which makes it `refused` — and appends the available URIs, so the correction is in the message.

## Prompts

The third primitive, and the one that fits this repo worst. Be warned that this section ends in a recommendation not to use it.

In [10]:
for p in session.list_prompts():
    print(f"{p.name}: {p.description}")
    for a in p.arguments or []:
        print(f"   arg {a.name!r} required={a.required}")

got = session.get_prompt("briefing", {"city": "Oslo"})
print()
for m in got.messages:
    print(f"  [{m.role}] {m.content.text}")

briefing: Ask for a plain-language weather briefing for a city.
   arg 'city' required=True

  [user] Give me a short weather briefing for Oslo. Mention whether I need a coat.


`prompts/get` returns **messages** — a conversation opener, filled in with arguments.

Now: where would that go in this repo? `01_foundations` writes its prompts itself, in the file. `02_agent_runtime` has a system prompt per component. Neither has a slot for *a prompt that arrives at runtime from a server*, because neither has the thing prompts are for.

What they are for is a **menu**. In a chat application, a server's prompts become slash commands — the user picks "briefing", is asked for a city, and the conversation starts there. The deciding actor is a person, at a keyboard, choosing between options they can see.

An autonomous agent has no such person. Handing its prompts to the *model* to pick from would make them tools with extra steps, and worse tools, since a prompt has no result to act on — it only seeds a conversation.

So the honest conclusion: **prompts are a UI affordance, and this repo has no UI.** There is a place for them the moment there is a person in the loop, and until then implementing support for them would be building a menu for nobody. Named here so that the absence is a decision rather than an oversight.

## What to do with resources, then

The one place they clearly belong is the case notebook 2 could not handle: data that has no name the model would think to ask for.

`04_memory` drew the line already, in its opening argument — memory is written by the agent from its own experience, a document corpus is bulk-loaded and the agent was never there. An MCP resource is the second of those, arriving over a wire. A server exposing a company handbook, a schema, a set of conventions is a **corpus**, and what a corpus needs is not a `read` call but retrieval: index it, search it, return the three paragraphs that matter.

Which is to say the useful shape is neither of the two options above:

In [11]:
registry = Registry()
for t in remote_tools(tools_only, prefix="weather"):
    registry.add(t)
registry.add(resource_reader(session), name="docs_read_resource")

print(registry.names)
print()
print("tools the model sees:", len(registry))
print("resources reachable through exactly one of them:",
      [str(r.uri) for r in session.list_resources()])

['weather_weather', 'weather_echo', 'weather_whoami', 'weather_raises', 'weather_described', 'weather_detailed', 'docs_read_resource']

tools the model sees: 7
resources reachable through exactly one of them: ['weather://stations', 'weather://history']


The registry holds tools. It always did — there is no `Registry.resources`, and adding one would be wrong, because a resource is not a thing the model selects between. What goes in the registry is the *one tool* that reaches them, and the resources stay a list the tool knows about.

For a corpus too large to list in a description, that tool stops being `read_resource(uri)` and becomes `search(query)` — which is `04_memory`'s `recall` with a different backing store, and two-stage retrieval for the same reason: return summaries and ids, let the model open what it wants. Nothing here changes that design; the resources are just where the documents came from.

## Where this leaves the chapter

In [12]:
session.__exit__(None, None, None)
tools_only.__exit__(None, None, None)
print("both sessions closed")

both sessions closed


Three primitives, and after taking each one seriously the ranking is not even close:

- **tools** carry the whole chapter. Everything in notebook 2 applies
- **resources** are real, and are a context-window decision rather than a data-access one. Handing a *small* one to an agent as a tool works well and restores both the size limit `03_tools` built and the failure classification notebook 2 built. Handing a *large* one that way only converts an overflow into a truncation, which is a safer failure and still a failure — for anything corpus-sized the shape is `04_memory`'s retrieval, not `read`
- **prompts** need a person choosing from a menu, and are worth skipping until there is one

Which retrospectively explains notebook 1's finding, and the reason `server.py` was left alone rather than having resources bolted on. It declares all three capabilities and offers only tools — and so does most of the ecosystem. Not because the other two are badly designed, but because **two of the three assume a host with a human in it**, and most things calling themselves MCP clients today are agents.

`04_mcp_server.ipynb` turns the chapter around: `Registry` exposed over MCP, `invoke()` running on the far side so `forge/kind` is real rather than inferred, and `verified()` finally in the one place it can work — the side where the world is.